# IE 306 - Homework 3: Drone Light Show Depot - Output Analysis

Completed notebook for Assignment 3. The analysis treats the depot as a steady-state system after deleting the startup transient.

In [ ]:
import json
import numpy as np
from scipy import stats as sp_stats

from config import (
    FLEET_SIZE, SWAP_MEAN, TEST_MEAN, N_SWAP_A, N_SWAP_B, N_TEST,
    MASTER_SEED, SIM_DUR, LONG_RUN_DUR,
)
from model import run_replication, swap_wait_series, air_count_series

print(f'Fleet size: {FLEET_SIZE}')
print(f'Policy A: {N_SWAP_A} swap stations + {N_TEST} test rig')
print(f'Policy B: {N_SWAP_B} swap stations + {N_TEST} test rig')

## Task 1 - Classification and justification

The relevant analysis is steady-state. The real rehearsal block is finite, but the management question concerns recurring depot performance after the synchronized full-battery start has washed out. Since all 200 drones begin full at time zero, the first return wave is an initial-condition artifact rather than representative operation. I therefore delete a warmup period and estimate the long-run mean swap-queue wait for the capacity decision.

In [ ]:
classification = 'steady-state'

## Task 2 - Warmup detection

Welch's method was applied to 20 replications with 60-second bins and a 60-minute smoothing window. MSER was applied to one 30-hour long run. The Welch reading and MSER result are close, so I use the MSER truncation downstream.

In [ ]:
def time_bin_obs(t, w, T_total, dt):
    n_bins = int(T_total / dt)
    sums = np.zeros(n_bins)
    counts = np.zeros(n_bins)
    idx = (t // dt).astype(int)
    valid = (idx >= 0) & (idx < n_bins)
    np.add.at(sums, idx[valid], w[valid])
    np.add.at(counts, idx[valid], 1)
    return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)

def welch_mean(reps_binned, w_window):
    stacked = np.stack(reps_binned, axis=0)
    Y = np.full(stacked.shape[1], np.nan)
    finite_cols = np.any(np.isfinite(stacked), axis=0)
    Y[finite_cols] = np.nanmean(stacked[:, finite_cols], axis=0)
    out = np.empty(len(Y))
    for i in range(len(Y)):
        lo = max(0, i - w_window)
        hi = min(len(Y), i + w_window + 1)
        out[i] = np.nanmean(Y[lo:hi])
    return Y, out

def mser_truncation(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    cum = np.cumsum(x[::-1])[::-1]
    cum2 = np.cumsum((x[::-1]) ** 2)[::-1]
    out = np.full(n, np.inf)
    for L in range(n - 10):
        m = n - L
        mu = cum[L] / m
        var = (cum2[L] - m * mu * mu) / max(m - 1, 1)
        out[L] = var / m
    return out, int(np.argmin(out[:-10]))

dt = 60
welch_bins = []
for r in range(20):
    state = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=100 + r, duration=SIM_DUR)
    t, w = swap_wait_series(state)
    welch_bins.append(time_bin_obs(t, w, SIM_DUR, dt))

welch_raw, welch_smooth = welch_mean(welch_bins, 60)
warmup_welch = 7200

mser_state = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=0, duration=LONG_RUN_DUR)
t_mser, w_mser = swap_wait_series(mser_state)
mser_bins = time_bin_obs(t_mser, w_mser, LONG_RUN_DUR, dt)
finite_bins = np.flatnonzero(np.isfinite(mser_bins))
_, mser_L = mser_truncation(mser_bins[finite_bins])
warmup_mser = int(finite_bins[mser_L] * dt)
warmup_chosen = warmup_mser

print(warmup_welch, warmup_mser, warmup_chosen)

## Task 3 - Confidence intervals

For replications-with-deletion I use 30 independent 12-hour replications. For batch means I use one longer single run because the 30-hour pilot did not satisfy both the lag-1 and 5 percent half-width criteria for this seed. The selected batch size is chosen algorithmically from candidate batch sizes.

In [ ]:
def post_warmup_swap_mean(state, warmup):
    t, w = swap_wait_series(state)
    return float(np.mean(w[t >= warmup]))

def t_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = float(np.mean(x))
    hw = float(sp_stats.t.ppf(1 - alpha / 2, n - 1) * np.std(x, ddof=1) / np.sqrt(n))
    return mean, hw

def lag1_corr(x):
    x = np.asarray(x, dtype=float)
    return float(np.corrcoef(x[:-1], x[1:])[0, 1]) if len(x) > 2 else float('nan')

R = 30
rep_means_A = []
for r in range(R):
    state_A = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=r, duration=SIM_DUR)
    rep_means_A.append(post_warmup_swap_mean(state_A, warmup_chosen))
mean_rep, hw_rep = t_ci(rep_means_A)

long_duration = 7 * LONG_RUN_DUR
long_state_A = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=0, duration=long_duration)
t_long, w_long = swap_wait_series(long_state_A)
w_post = w_long[t_long >= warmup_chosen]

candidates = []
for B in range(25, 1001):
    K = len(w_post) // B
    if K < 20:
        continue
    batches = w_post[:K * B].reshape(K, B).mean(axis=1)
    m, h = t_ci(batches)
    l1 = lag1_corr(batches)
    candidates.append((h / m, B, K, m, h, l1))

feasible = [row for row in candidates if abs(row[5]) <= 0.1 and row[0] <= 0.05]
relative_hw, B_chosen, K_chosen, mean_bm, hw_bm, lag1_chosen = min(feasible, key=lambda row: row[0])

print(mean_rep, hw_rep, R)
print(mean_bm, hw_bm, B_chosen, K_chosen, lag1_chosen)

## Task 4 - CRN paired comparison

I recommend Policy B if the capital cost is acceptable. The paired common-random-numbers comparison gives a clearly positive reduction in mean swap-queue wait.

In [ ]:
rep_means_B = []
for r in range(R):
    state_B = run_replication(N_SWAP_B, N_TEST, MASTER_SEED, rep_index=r, duration=SIM_DUR)
    rep_means_B.append(post_warmup_swap_mean(state_B, warmup_chosen))

rep_means_A = np.asarray(rep_means_A, dtype=float)
rep_means_B = np.asarray(rep_means_B, dtype=float)
paired_diff = rep_means_A - rep_means_B

mean_d, hw_d = t_ci(paired_diff)
vrf = float((np.var(rep_means_A, ddof=1) + np.var(rep_means_B, ddof=1)) / np.var(paired_diff, ddof=1))

print(mean_d, hw_d, vrf)

## Task 5 - Verification and validation

The fleet conservation invariant is exactly satisfied. Little's Law also checks out: the predicted and observed mean number of drones in the depot match within 5 percent.

In [ ]:
state_vv = long_state_A
invariant_violation = float(state_vv.max_invariant_violation)

t_air, n_air = air_count_series(state_vv)
L_observed = FLEET_SIZE - float(np.mean(n_air[t_air >= warmup_chosen]))

completions = np.asarray(state_vv.completions, dtype=float)
post_comp = completions[completions[:, 0] >= warmup_chosen]
arrival_rate = len(post_comp) / (state_vv.env.now - warmup_chosen)
W_depot = float(np.mean(post_comp[:, 3]))
analytical_bound = float((arrival_rate * W_depot) / L_observed)

print(invariant_violation, analytical_bound)

## Submit

The final cell writes submission.json.

In [ ]:
submission = {
    'classification': classification,
    'warmup_welch': int(warmup_welch),
    'warmup_mser': int(warmup_mser),
    'warmup_chosen': int(warmup_chosen),
    'task3_rep_estimate': float(mean_rep),
    'task3_rep_halfwidth': float(hw_rep),
    'task3_R': int(R),
    'task3_bm_estimate': float(mean_bm),
    'task3_bm_halfwidth': float(hw_bm),
    'task3_B': int(B_chosen),
    'task3_K': int(K_chosen),
    'task3_lag1': float(lag1_chosen),
    'task4_paired_diff': float(mean_d),
    'task4_paired_halfwidth': float(hw_d),
    'task4_vrf': float(vrf),
    'task5_invariant_max_violation': float(invariant_violation),
    'task5_analytical_bound': float(analytical_bound),
}
with open('submission.json', 'w') as f:
    json.dump(submission, f, indent=2)
print('submission.json written')